In [444]:
import numpy as np
import pandas as pd
from numpy import nan
from sklearn.preprocessing import  StandardScaler, OneHotEncoder, MinMaxScaler
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.datasets import make_classification 
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA, TruncatedSVD
from nltk.stem import SnowballStemmer
from sklearn import linear_model, datasets
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC, LinearSVR
from sklearn.feature_selection import SelectKBest, SelectFpr, chi2, mutual_info_classif, f_classif
from sklearn.linear_model import SGDClassifier
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import SnowballStemmer
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
import re
import string
from bs4 import BeautifulSoup
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import confusion_matrix

In [445]:
df = pd.read_csv("winter_project_2026/development.csv")
eva= pd.read_csv("winter_project_2026/evaluation.csv")

In [446]:
df.drop_duplicates(inplace=True)
df = df.dropna()
test_ids = eva['Id'].copy()
eva['article'].fillna("", inplace=True)

/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_50055/2900358585.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  eva['article'].fillna("", inplace=True)


In [447]:
# unisco le colonne titolo e articolo in una colonna unica raddoppiando l'importanza del titolo
df['text'] = df['source'] + ' ' + df['source'] + ' ' + df['title'] + ' ' + df['title'] + ' ' + df['title'] + ' ' + df['article'] + ' ' + df['article']
eva['text'] = eva['source'] + ' ' + eva['source'] + ' ' + eva['title'] + ' ' + eva['title'] + ' ' + eva['title'] + ' ' + eva['article'] + ' ' + eva['article']

df.columns

Index(['Id', 'source', 'title', 'article', 'page_rank', 'timestamp', 'label',
       'text'],
      dtype='object')

In [448]:
weekdays = []
daytime = []
for day in df['timestamp']:
    if day == '0000-00-00 00:00:00':
        week_day = -1
        hour_day = -1
    else:
        week_day = pd.Timestamp(day).day_of_week
        hour = pd.Timestamp(day).hour
        if hour > 5 and hour <= 14:
            hour_day = 1        #morning
        elif hour > 14 and hour <= 21:
            hour_day = 2        #afternoon
        elif (hour > 21 and hour <= 23) or (hour >= 0 and hour <= 5):
            hour_day = 3        #night

    daytime.append(hour_day)
    weekdays.append(week_day)

df['day_of_week'] = weekdays
df['moment_of_day'] = daytime


weekdays_eva = []
daytime_eva = []
for day in eva['timestamp']:
    if day == '0000-00-00 00:00:00':
        week_day = -1
        hour_day = -1
    else:
        week_day = pd.Timestamp(day).day_of_week
        hour = pd.Timestamp(day).hour
        if hour > 5 and hour <= 14:
            hour_day = 1        #morning
        elif hour > 14 and hour <= 21:
            hour_day = 2        #afternoon
        elif (hour > 21 and hour <= 23) or (hour >= 0 and hour <= 5):
            hour_day = 3        #night

    daytime_eva.append(hour_day)
    weekdays_eva.append(week_day)

eva['day_of_week'] = weekdays_eva
eva['moment_of_day'] = daytime_eva

In [ ]:
y = df['label']
df.drop('label', inplace=True, axis=1)
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=42, stratify=y, shuffle=True)


In [ ]:
encoder = OneHotEncoder(min_frequency=70, handle_unknown='ignore')

vectorizer = TfidfVectorizer(
        max_features=60000,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=3)

vectorizer_title = TfidfVectorizer(
        max_features=10000,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=3)

vectorizer2 = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    min_df=5,
    max_features=30000
)

preprocessor = ColumnTransformer(transformers=[
    ('source', encoder, ['source', 'day_of_week', 'moment_of_day']),
    ('text',vectorizer, 'text'),
    ('text_char',vectorizer2, 'text'),
], remainder='drop', n_jobs=-1)



In [ ]:
X_train.shape

In [ ]:
X_train_final = preprocessor.fit_transform(X_train)
X_test_final = preprocessor.transform(X_test)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


In [ ]:
svm_clf = SGDClassifier(
    loss='log_loss', 
    penalty='l2',
    alpha=1e-5,
    max_iter=2000,
    tol=1e-6,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
clf = LogisticRegression(
    C=1,                       
    solver='saga',        
    penalty='elasticnet',           
    max_iter=1500,              
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
    multi_class='multinomial',
    l1_ratio=0.5
)

voting_clf = VotingClassifier(
    estimators=[ 
        ('svm', svm_clf), 
        ('lr', clf)
    ], n_jobs=-1,
    voting='soft'
)

svm_clf.fit(X_train_final, y_train)
y_pred = svm_clf.predict(X_test_final)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


In [ ]:
#submission = pd.DataFrame({
#    'Id': test_ids,    # Usiamo gli ID salvati all'inizio
#    'Predicted': y_pred # O 'label' o 'Category' a seconda delle regole della gara
#})
#
#submission.to_csv('submission.csv', index=False)
#print("Fatto! File pronto.")

Fatto! File pronto.
